# Predicción de devoluciones -- Proyecto de machine learning para reducción de devoluciones en un e-commerce

## Contexto de Negocio.

### Contexto Comercial y Problemática

Actualmente, la infraestructura logística de la empresa está operando por debajo de los estándares esperados, situación que se refleja principalmente en una alta tasa de devoluciones. En un entorno de e-commerce, donde los tiempos de entrega y la confiabilidad del servicio tienen un impacto directo en la experiencia del cliente, estas ineficiencias representan un problema relevante. Los retrasos, los pedidos que no llegan correctamente a destino y las devoluciones no solo afectan la satisfacción y fidelización de los clientes, sino que también generan sobrecostos asociados al transporte, almacenamiento, gestión de devoluciones y reprocesamiento de pedidos, impactando directamente en la rentabilidad de la empresa.

Ante esta situación, resulta necesario comprender cuáles son los principales factores que están afectando el desempeño de la operación logística. Variables como la ubicación de los clientes y depósitos, la categoría de los productos, la prioridad de los pedidos, los tiempos de procesamiento y las características de cada envío pueden permitir identificar patrones que no son evidentes a simple vista. El análisis de estos datos permitirá detectar posibles puntos críticos dentro de la cadena de distribución y determinar en qué etapas se concentran las mayores dificultades. Para ello, utilizaremos un csv que cuenta con 50.000 registros y 30 columnas, cubriendo pedidos registrados entre el 1 de enero y el 31 de diciembre de 2026.

### Objetivo del análisis

El objetivo de este análisis es explorar los datos de envíos recientes para identificar los principales cuellos de botella en la cadena de distribución, analizar su relación con las devoluciones y los retrasos , y detectar oportunidades concretas de mejora. A partir de estos hallazgos, se propondrá un plan de acción basado en datos que permita optimizar los tiempos de entrega, reducir las devoluciones logísticas y mejorar la eficiencia general del sistema de distribución. De esta manera, se busca no solo disminuir los costos operativos, sino también contribuir a una experiencia de compra más confiable y consistente para los clientes.

### Planteo de Hipótesis

H1 (Categoría): Las categorías de mayor rotación/sensibilidad (como Fashion y Electronics) registran una tasa de devolución superior a categorías como Toys.

H2 (Demora): Los envíos con retraso (late_delivery = Yes o mayor delivery_delay_days) concentran una proporción significativamente mayor de solicitudes de devolución.

H3 (Prioridad): A menor prioridad del pedido (Low vs. Urgent), mayor es la tasa de devolución observada.

H4 (Pago): El método de pago muestra diferencias menores en la tasa de devolución en comparación con variables de entrega y calificación.

H5 (Variables críticas): Un subconjunto de variables de entrega, satisfacción y características del pedido concentra el mayor poder predictivo para el modelo.

## Data Adquisition

### Importación de Librerías.

In [9]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

### Recolección de Datos

In [3]:
# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Leer el archivo CSV
ruta = '/content/drive/MyDrive/DATA-SCIENCE/Data-Science-II/Dataset_Envios.csv'
df = pd.read_csv(ruta)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Data Understanding

### Diccionario de datos

1. **order_id** --> Identificador único alfanumérico del pedido (ej. ORD-000001).
2. **order_date** --> Fecha en la que se realizó el pedido (formato AÑO-MES-DÍA).
3. **customer_id** --> Identificador único del comprador.
4. **customer_segment** --> Segmento comercial al que pertenece el cliente (Consumer, Small Business, Premium y Enterprise).
5. **customer_city** --> Ciudad de residencia o destino del cliente.
6. **customer_country** --> País de residencia o destino del cliente.
7. **warehouse_id** --> Identificador único del almacén de origen donde se preparó el pedido.
8. **warehouse_city** --> Ciudad donde está ubicado el almacén de origen.
9. **product_category** --> Categoría principal del producto vendido (ej. Home & Kitchen, Beauty).
10. **product_weight_kg** --> Peso del producto expresado en kilogramos.
11. **order_value_usd** --> Valor total monetario del pedido expresado en dólares estadounidenses (USD).
12. **shipping_method** --> Modalidad de envío seleccionada por el cliente o sistema (ej. Standard, International).
13. **carrier** --> Empresa de logística o transportista encargada de llevar el paquete.
14. **distance_km** --> Distancia total del trayecto de envío en kilómetros.
15. **promised_delivery_days** --> Cantidad de días estimados o prometidos inicialmente para la entrega.
16. **actual_delivery_days** --> Cantidad de días reales que transcurrieron hasta que el paquete fue entregado.
17. **delivery_status** --> Estado del envío al momento del registro (Delivered, In Transit, Delayed, Failed Delivery).
18. **late_delivery** --> Indicador binario (Yes / No) que señala si la entrega superó la fecha prometida.
19. **delivery_delay_days** --> Cantidad de días exactos de retraso. Es 0 si se entregó a tiempo o antes.
20. **shipping_cost_usd** --> Costo logístico del envío expresado en dólares estadounidenses (USD).
21. **package_size** --> Clasificación del tamaño físico del paquete (Small < Medium < Large < Oversized).
22. **payment_method** --> Medio de pago utilizado para abonar el pedido (ej. Digital Wallet, Debit Card).
23. **order_priority** --> Nivel de urgencia o prioridad asignado al procesamiento del pedido (Low < Normal < High < Urgent).
24. **weather_condition** --> Condición climática predominante registrada durante el proceso de envío (ej. Snow, Cloudy).
25. **customer_rating** --> Calificación otorgada por el cliente al servicio o producto, en una escala de 1 a 5.
26. **return_requested** --> Indicador binario (Yes / No) que señala si el cliente solicitó devolver el producto.
27. **return_reason** --> Motivo declarado por el cliente para la devolución (ej. Late Delivery, Product Defect). Es nulo si no hay devolución.
28. **delivery_attempts** --> Número de veces que el transportista intentó entregar el paquete en el domicilio.
29. **warehouse_processing_hours** --> Cantidad de horas netas que tardó el almacén en preparar y despachar el pedido desde su ingreso.
30. **tracking_status** --> Estado de seguimiento público del paquete (Delivered, In Transit, Exception, Out for Delivery).


### Exploración de los datos

In [11]:
# Primero generamos una copia del original para no modificar nada de la información entrante.
df_original = df.copy()

In [12]:
# Vemos el shape de nuestro dataset para entender con lo que estamos trabajando
df.shape
# Vemos que son 50 mil filas y 30 columnas.

(50000, 30)

In [14]:
# Veamos un vistazo a las variables
df.head(5)

,order_id,order_date,customer_id,customer_segment,customer_city,customer_country,warehouse_id,warehouse_city,product_category,product_weight_kg,...,package_size,payment_method,order_priority,weather_condition,customer_rating,return_requested,return_reason,delivery_attempts,warehouse_processing_hours,tracking_status
0,ORD-000001,2026-11-15,CUS-002353,Small Business,Tokyo,Japan,WH-004,Toronto,Home & Kitchen,8.23,...,Large,Digital Wallet,Normal,Snow,3,Yes,Late Delivery,2,25.1,Delivered
1,ORD-000002,2026-04-08,CUS-003258,Consumer,Vancouver,Canada,WH-007,Paris,Beauty,0.10,...,Small,Debit Card,Normal,Cloudy,5,No,NaN,1,25.9,Delivered
2,ORD-000003,2026-01-05,CUS-001221,Consumer,Houston,United States,WH-005,London,Home & Kitchen,5.51,...,Large,Debit Card,Low,Snow,4,Yes,Product Defect,1,31.5,Delivered
3,ORD-000004,2026-05-29,CUS-001717,Small Business,Delhi,India,WH-002,Los Angeles,Grocery,3.75,...,Large,Debit Card,Normal,Snow,3,No,NaN,1,12.6,In Transit
4,ORD-000005,2026-02-22,CUS-000716,Small Business,Nagoya,Japan,WH-006,Berlin,Beauty,0.10,...,Small,Credit Card,Normal,Storm,2,No,NaN,1,24.9,Delivered


In [16]:
# Con el info podemos ver la cantidad de nulos y los tipos de las columnas de nuestro dataset
df.info()
# Esta información nos da pie a localizar que la columna return_reason tiene registros nulos.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   order_id                    50000 non-null  object 
 1   order_date                  50000 non-null  object 
 2   customer_id                 50000 non-null  object 
 3   customer_segment            50000 non-null  object 
 4   customer_city               50000 non-null  object 
 5   customer_country            50000 non-null  object 
 6   warehouse_id                50000 non-null  object 
 7   warehouse_city              50000 non-null  object 
 8   product_category            50000 non-null  object 
 9   product_weight_kg           50000 non-null  float64
 10  order_value_usd             50000 non-null  float64
 11  shipping_method             50000 non-null  object 
 12  carrier                     50000 non-null  object 
 13  distance_km                 500

In [20]:
# Por lo tanto veamos que tan relevantes son esos nulos.
df.isnull().sum()
nulos = (41375/50000) * 100
print(f"return_reason tiene un {round(nulos)}% de nulos")

return_reason tiene un 83% de nulos


In [22]:
# Veamos si existen duplicados
df.duplicated().sum()
# Confirmamos la presencia nula de registros duplicados.

np.int64(0)

In [24]:
# Por último, quiero ver como se distribuyen mis datos para tener una base antes de hacer el data Wrangling
df.describe().T
# Reconozco que varias columnas presentan un sesgo negativo. Hay que analizarlo pero nos da un indicio de como se mueven nuestras variables númericas.

,count,mean,std,min,25%,50%,75%,max
product_weight_kg,50000.0,2.255028,3.694952,0.10,0.320,0.93,2.6100,52.19
order_value_usd,50000.0,177.917107,291.600047,10.00,28.720,76.84,173.1625,3530.73
distance_km,50000.0,2496.778802,1774.633152,40.00,1086.775,2182.80,3592.4750,9553.60
promised_delivery_days,50000.0,7.197700,3.680229,0.00,4.000,6.00,11.0000,15.00
actual_delivery_days,50000.0,8.345940,4.644353,0.00,5.000,8.00,12.0000,27.00
delivery_delay_days,50000.0,1.281080,1.679527,0.00,0.000,1.00,2.0000,13.00
shipping_cost_usd,50000.0,120.515538,108.680030,4.05,40.640,82.97,167.4225,500.00
customer_rating,50000.0,3.371840,0.998777,1.00,3.000,3.00,4.0000,5.00
delivery_attempts,50000.0,1.169860,0.390756,1.00,1.000,1.00,1.0000,3.00
warehouse_processing_hours,50000.0,14.412430,9.924904,1.00,6.600,11.90,20.7000,48.00


## Data Wrangling